# POEM dual-T4 Kaggle training

Attach two Kaggle datasets before running:

1. `POEM-BASE`: a mirror of this GitHub repository
2. `Beautiful-Motifs-CC-BY-NC-SA`: the MIDI dataset

Set the Kaggle accelerator to **GPU T4 x2**. The workflow pretokenizes short motifs, trains variants D/C/E/B/A, generates five MIDI samples per completed variant, and uploads organized artifacts to a Hugging Face model repo named `POEM-BASE` under your namespace.

In [ ]:
!nvidia-smi
!python --version
!pip install -q pretty_midi mido huggingface_hub

In [ ]:
from pathlib import Path
import os, shutil, json

def find_dataset_dir(candidates):
    for path in candidates:
        p = Path(path)
        if p.exists():
            return p
    raise FileNotFoundError(candidates)

INPUT = Path('/kaggle/input')
repo_src = find_dataset_dir([
    INPUT / 'poem-base',
    INPUT / 'POEM-BASE',
    INPUT / 'poem',
    INPUT / 'POEM',
])
repo_dst = Path('/kaggle/working/POEM-BASE')
if repo_dst.exists():
    shutil.rmtree(repo_dst)
shutil.copytree(repo_src, repo_dst, ignore=shutil.ignore_patterns('.git', '__pycache__', '*.pyc'))
os.chdir(repo_dst)
print('Repo:', repo_dst)

data_dir = None
for p in INPUT.rglob('*'):
    if p.is_dir() and (p / 'MIDIs').exists():
        data_dir = p
        break
if data_dir is None:
    raise FileNotFoundError('Could not find Beautiful-Motifs dataset folder containing MIDIs/')
print('Data:', data_dir)

In [ ]:
from getpass import getpass
import os

HF_TOKEN = getpass('Paste a Hugging Face token with write access to your private POEM-BASE repo: ')
os.environ['HF_TOKEN'] = HF_TOKEN
HF_NAMESPACE = input('Hugging Face username or org: ').strip()
HF_REPO_ID = f'{HF_NAMESPACE}/POEM-BASE'
print('HF repo:', HF_REPO_ID)

In [ ]:
# Tune these if the 12-hour session is tight. The script stops before starting a new variant once max_hours is reached.
EPOCHS = 40
BATCH_SIZE = 256
MAX_HOURS = 11.75
VARIANTS = 'D C E B A'

cmd = f"""
python -u scripts/kaggle_train_all.py \
  --repo_dir /kaggle/working/POEM-BASE \
  --data_dir '{data_dir}' \
  --hf_repo_id '{HF_REPO_ID}' \
  --private \
  --epochs {EPOCHS} \
  --batch_size {BATCH_SIZE} \
  --variants {VARIANTS} \
  --max_hours {MAX_HOURS} \
  --val_interval 2000 \
  --checkpoint_interval_steps 5000 \
  --checkpoint_interval_minutes 20 \
  --num_workers 2
"""
print(cmd)
!{cmd}

In [ ]:
from pathlib import Path
summary = Path('/kaggle/working/comparison/summary.json')
if summary.exists():
    print(summary.read_text()[:4000])
else:
    print('No comparison summary found yet.')
!find /kaggle/working -maxdepth 3 -type f | sed 's#/kaggle/working/##' | sort | head -200